In [3]:
import pandas as pd

excel_path = "Yamuna_Under_Drain_Analysis.xlsx"
df_stations = pd.read_excel(excel_path, sheet_name="Yamuna_Delhi_Stations")

# Exact column names dekhne ke liye (extra spaces ke saath)
print(df_stations.columns.tolist())

['Unnamed: 0', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12']


In [5]:
import pandas as pd

excel_path = "Yamuna_Under_Drain_Analysis.xlsx"
df_stations = pd.read_excel(excel_path, sheet_name="Yamuna_Delhi_Stations")

print("Columns:", df_stations.columns.tolist())
print(df_stations.head(3))

Columns: ['Unnamed: 0', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12']
   Unnamed: 0                                         Unnamed: 1 Unnamed: 2  \
0         NaN  CPCB Monitoring Stations: River Yamuna (Delhi ...        NaN   
1         NaN  Real Water Quality Records: Physico-Chemical &...        NaN   
2         NaN                                                NaN        NaN   

  Unnamed: 3 Unnamed: 4 Unnamed: 5 Unnamed: 6 Unnamed: 7 Unnamed: 8  \
0        NaN        NaN        NaN        NaN        NaN        NaN   
1        NaN        NaN        NaN        NaN        NaN        NaN   
2        NaN        NaN        NaN        NaN        NaN        NaN   

  Unnamed: 9 Unnamed: 10 Unnamed: 11 Unnamed: 12  
0        NaN         NaN         NaN         NaN  
1        NaN         NaN         NaN         NaN  
2        NaN         NaN         NaN         NaN  

In [6]:
import pandas as pd
import numpy as np

excel_path = "Yamuna_Under_Drain_Analysis.xlsx"

# 1. Load Excel Sheets (Upar ke title rows skip karke)
df_stations = pd.read_excel(excel_path, sheet_name="Yamuna_Delhi_Stations", skiprows=3).dropna(axis=1, how='all')
df_drains = pd.read_excel(excel_path, sheet_name="Delhi_Drains_Discharge", skiprows=3).dropna(axis=1, how='all')
df_trends = pd.read_excel(excel_path, sheet_name="Yearly_Trends_2018_2024", skiprows=3).dropna(axis=1, how='all')

# Column names clean karo
df_stations.columns = df_stations.columns.str.strip().str.replace('\n', ' ')
print("Actual Columns:", df_stations.columns.tolist())

# 2. Handling Missing & Censored Values ("BDL" -> 0.1)
do_col = [c for c in df_stations.columns if "DO" in c and "Min" in c][0]
bod_col = [c for c in df_stations.columns if "BOD" in c and ("Max" in c or "Peak" in c)][0]

df_stations[do_col] = pd.to_numeric(df_stations[do_col].replace("BDL", 0.1), errors='coerce')

# 3. Feature Engineering: BOD Pollution Multiplier (Safe limit = 3 mg/L)
df_stations["BOD_Violation_Factor"] = (df_stations[bod_col] / 3.0).round(1)

print(df_stations[[df_stations.columns[1], do_col, bod_col, "BOD_Violation_Factor"]].dropna())

Actual Columns: ['Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12']


IndexError: list index out of range

In [7]:
import pandas as pd
import numpy as np

excel_path = "Yamuna_Under_Drain_Analysis.xlsx"

# Raw sheet load karke automatically header row ('Station Code') dhoondho
raw_df = pd.read_excel(excel_path, sheet_name="Yamuna_Delhi_Stations", header=None)
header_row = raw_df[raw_df.apply(lambda r: r.astype(str).str.contains("Station Code", case=False).any(), axis=1)].index[0]

# Asli header ke saath data load karo
df_stations = pd.read_excel(excel_path, sheet_name="Yamuna_Delhi_Stations", skiprows=header_row).dropna(axis=1, how='all')
df_stations.columns = df_stations.columns.str.strip().str.replace('\n', ' ')

print("Exact Columns:", df_stations.columns.tolist())
print(df_stations.head(3))


Exact Columns: ['Station Code', 'Monitoring Station Name', 'Segment Type', 'DO Min (mg/L)', 'DO Max (mg/L)', 'BOD Min (mg/L)', 'BOD Max (mg/L)', 'BOD Mean (mg/L)', 'COD Min (mg/L)', 'COD Max (mg/L)', 'COD Mean (mg/L)', 'Water Quality Class']
   Station Code              Monitoring Station Name             Segment Type  \
0          1120        Yamuna at Palla (Delhi Entry)  Upstream Clean Baseline   
1          5098          Yamuna at Wazirabad Barrage         Pre-Urban Inflow   
2          5099  Yamuna at ISBT Kashmiri Gate Bridge    Post Najafgarh Inflow   

   DO Min (mg/L)  DO Max (mg/L)  BOD Min (mg/L)  BOD Max (mg/L)  \
0            6.1            9.6             1.3               4   
1            4.0            7.9             3.0               9   
2            0.3            6.4            11.0              51   

   BOD Mean (mg/L)  COD Min (mg/L)  COD Max (mg/L)  COD Mean (mg/L)  \
0              2.6              18              44             25.5   
1              6.6    

In [8]:
import pandas as pd
import numpy as np

excel_path = "Yamuna_Under_Drain_Analysis.xlsx"

# Helper function jo kisi bhi sheet ka exact header row automatically dhoondh kar load kare
def load_clean_sheet(sheet_name, keyword):
    raw_df = pd.read_excel(excel_path, sheet_name=sheet_name, header=None)
    header_row = raw_df[raw_df.apply(lambda r: r.astype(str).str.contains(keyword, case=False).any(), axis=1)].index[0]
    df = pd.read_excel(excel_path, sheet_name=sheet_name, skiprows=header_row).dropna(axis=1, how='all')
    df.columns = df.columns.str.strip().str.replace('\n', ' ')
    return df

# 1. Load all 3 Excel Sheets cleanly
df_stations = load_clean_sheet("Yamuna_Delhi_Stations", "Station Code")
df_drains = load_clean_sheet("Delhi_Drains_Discharge", "Drain")
df_trends = load_clean_sheet("Yearly_Trends_2018_2024", "Year")

# 2. Handling Missing & Censored Values ("BDL" -> 0.1)
df_stations["DO Min (mg/L)"] = pd.to_numeric(df_stations["DO Min (mg/L)"].replace("BDL", 0.1), errors='coerce')

# 3. Feature Engineering: BOD Pollution Multiplier (Safe limit = 3 mg/L)
df_stations["BOD_Violation_Factor"] = (df_stations["BOD Max (mg/L)"] / 3.0).round(1)

# Clean result check
print(df_stations[["Monitoring Station Name", "DO Min (mg/L)", "BOD Max (mg/L)", "BOD_Violation_Factor", "Water Quality Class"]])

                Monitoring Station Name  DO Min (mg/L)  BOD Max (mg/L)  \
0         Yamuna at Palla (Delhi Entry)            6.1               4   
1           Yamuna at Wazirabad Barrage            4.0               9   
2   Yamuna at ISBT Kashmiri Gate Bridge            0.3              51   
3                  Yamuna at ITO Bridge            0.3              66   
4                  Yamuna at Nizamuddin            0.3              53   
5   Yamuna at Okhla Bridge (Agra Canal)            0.3              60   
6  Yamuna at Okhla after Shahdara Drain            0.3              85   

   BOD_Violation_Factor                Water Quality Class  
0                   1.3          Class B (Bathing Quality)  
1                   3.0     Class C (Moderate Degradation)  
2                  17.0             Toxic (Septic Stretch)  
3                  22.0             Toxic (Septic Stretch)  
4                  17.7             Toxic (Septic Stretch)  
5                  20.0  Hazardous (Zero 